# RAG over Documents

**Retrieval-Augmented Generation (RAG)** lets an AI answer questions using *your* content instead of only what it learned in training. The content is converted into vector embeddings, the passages closest in meaning to a question are retrieved, and the model writes an answer grounded in those passages.

With AIDB all of this happens inside Postgres, using AI Pipelines, a custom SQL tool, and an agent.

## What you will build

| Part | Source of knowledge | Pipeline |
|---|---|---|
| **1. Knowledge Base on a database table** | 12 short nutrition notes in a table | One embedding per row |
| **2. Knowledge Base on PDF documents** | Two IRS tax publications (PDF) | Parse, then chunk, then embed |
| **3. An agent with both tools** | Both of the above | The agent chooses the right tool for each question |
| **4. Logging** | Every conversation | Stored automatically in Postgres tables |

For each source the recipe is the same: **pipeline, similarity search, custom tool, agent**.

## Prerequisites

- A Postgres database with AIDB installed. Part 2 also needs permission to restart it.
- An Anthropic (Claude) API key.
- Internet access from the database host to download the IRS PDFs in Part 2.

# Part 1: Knowledge Base on a database table

The first demo will use a text field in a table to create a knowledge base.

## Claude API Key

This demo creates agents that use a Claude Sonnet model, so an Anthropic API key is required. Run the next cell and paste your key when prompted. It is held in memory only and passed to the database when the model is registered.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

## Connect to the database

Creates a SQLAlchemy engine for the `aidb` database and attaches the `%sql` magics to it. `pool_pre_ping=True` checks connections before reuse, and named parameters are enabled so `%%sql` cells can use Python variables such as `:creds`.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Register Claude Sonnet

`aidb.create_model` registers the Claude Sonnet 5 model inside the database under the name `claude_model_rag_over_documents`. All of the agents in this notebook use it to reason about questions and write answers. Any earlier registration with the same name is deleted first so the notebook can be re-run.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'claude_model_rag_over_documents';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

## Create Demo Table

This table, `public.food_notes`, holds 12 short tips on nutrition, fitness, food safety, dietary needs, and cooking. Each row has an `id`, a `title`, a `category`, and the text in `body`.

Note: the table contains unstructured text. The length of the text is relatively short (a sentence or two), so chunking is not needed. Each row will become a single embedding.

In [ ]:
%%sql

DROP TABLE IF EXISTS public.food_notes;

CREATE TABLE public.food_notes (
    id       int PRIMARY KEY,
    title    text,
    category text,
    body     text
);

INSERT INTO public.food_notes VALUES
(1,'Iron and plant sources','nutrition','Spinach, lentils and tofu are good plant sources of iron. Pair them with vitamin C foods such as citrus or peppers to improve absorption.'),
(2,'Reducing sodium','nutrition','High sodium intake raises blood pressure. Cut back on processed meats, canned soups and salty snacks, and season with herbs, lemon or garlic instead.'),
(3,'Fibre and digestion','nutrition','Oats, beans, apples and whole grains are rich in fibre. Soluble fibre helps lower cholesterol, insoluble fibre keeps digestion regular.'),
(4,'Protein for recovery','fitness','After exercise, eggs, Greek yogurt, chicken or tofu supply protein that supports muscle repair. Aim for roughly 20 to 30 grams per meal.'),
(5,'Safe chicken handling','safety','Cook poultry to an internal temperature of 74 C (165 F). Keep raw chicken separate from salad ingredients to avoid cross-contamination.'),
(6,'Refrigeration rules','safety','Refrigerate leftovers within two hours. Cooked rice left at room temperature can grow Bacillus cereus, so cool and chill it quickly.'),
(7,'Gluten-free swaps','dietary','For a gluten-free diet use rice, quinoa, buckwheat or certified gluten-free oats. Check labels on sauces and soups for hidden wheat.'),
(8,'Lactose intolerance','dietary','People with lactose intolerance can often tolerate hard cheese and yogurt. Lactose-free milk or fortified oat and soy drinks provide calcium.'),
(9,'Healthy fats','nutrition','Olive oil, avocado, nuts and oily fish provide unsaturated fats. Salmon and sardines are also rich in omega-3 fatty acids.'),
(10,'Sugar and fruit','nutrition','Whole fruit contains sugar but also fibre and water, which slow absorption. Fruit juice and sweetened drinks spike blood sugar faster.'),
(11,'Meal prep basics','cooking','Batch cook grains and proteins on Sunday, store in airtight containers, and eat within four days. Freeze extra portions.'),
(12,'Calcium and bone health','nutrition','Dairy, fortified plant milks, kale and canned sardines with bones supply calcium. Vitamin D helps the body absorb it.');

## Create AI Pipeline

AI Pipelines can ingest, prepare, and vectorize data for semantic retrieval and RAG applications. They integrate pgvector search capabilities and can keep the knowledge base synchronized with your source data automatically.

This particular pipeline creates a vector embedding per row of the table. It is configured with:

- `source`: the table to read (`food_notes`).
- `source_key_column` and `source_data_column`: the column that identifies each row (`id`) and the column holding the text to embed (`body`).
- A single `KnowledgeBase` step using the `bge-small-en-v1.5-f16` embedding model.
- `auto_processing => 'Disabled'`, meaning it only runs when asked. `aidb.run_pipeline` then generates the embeddings.

The result is a table named `pipeline_food_kb` that is searched in the next step. If the notes change later, run the pipeline again to refresh it.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'food_kb';
    
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.pipelines WHERE name = v_name) THEN
        PERFORM aidb.delete_pipeline(v_name);
    END IF;

    PERFORM aidb.create_pipeline(
        name               => v_name,
        source             => 'food_notes',
        source_key_column  => 'id',
        source_data_column => 'body',
        step_1             => 'KnowledgeBase',
        step_1_options     => aidb.knowledge_base_config(
            model       => 'bge-small-en-v1.5-f16',
            data_format => 'Text'
        ),
        auto_processing    => 'Disabled'
    );
    
    PERFORM aidb.run_pipeline(v_name);
END    
$do$;

## Similarity Search

Use the meanings of the words to find the most similar vector embeddings.

`aidb.retrieve_text` takes the knowledge base table, a question, and the number of results (here, 3). The question is turned into an embedding, and the closest notes are returned with a `distance` score. Notice that the match is based on meaning and not on shared keywords. Expect the protein-for-recovery note to rank high for *"what should I eat to build muscle?"*.

In [ ]:
%%sql
SELECT key, left(value, 70) AS snippet, distance::numeric(5,3) AS distance
FROM aidb.retrieve_text('public.pipeline_food_kb',
                        'what should I eat to build muscle?', 3);

## Create Custom Tool

The similarity search is nice, but using an agent with a custom tool gives you the ability to ask questions and have a conversation.

A **SQL tool** wraps a SQL statement so an agent can call it:

- `description` is what the agent reads to decide *when* to use the tool, so it says what the tool covers and when to use it.
- `${query}` and `${top_k}` are placeholders that the agent fills in. They are declared in `params`.
- `read_only => true` marks the tool as safe because it only reads data.

This custom tool uses the vector embeddings created above and will be used later by an agent.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'search_food';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.sql_tool_registry WHERE name = v_name) THEN
        PERFORM aidb.delete_tool(v_name);
    END IF;

    PERFORM aidb.create_sql_tool(
        name          => v_name,
        description   => 'Semantic search over short food and nutrition notes (nutrition, safety, dietary, cooking tips). Returns the most relevant notes. Use for questions about nutrition advice, food safety, or dietary needs.',
        sql_statement => $$
            SELECT key AS note_id,
                   distance::numeric(5,3) AS distance,
                   value AS passage
            FROM aidb.retrieve_text('public.pipeline_food_kb'::text, ${query}::text, ${top_k}::int)
        $$,
        params        => aidb.tool_params(
            aidb.tool_param('query', 'TEXT', 'The question or search phrase.'),
            aidb.tool_param('top_k', 'INT', 'Number of notes to return, e.g. 3.')
        ),
        read_only     => true
    );
END
$do$;

## Create Agent with Custom Tool

This agent uses the custom tool just created. Its instructions tell it to rely on `search_food` when answering, and the tool is the only thing it can call. This enables users to have a conversation about the private data in your database that the tool provides.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'demo_agent_nutrition';
BEGIN
    PERFORM aidb.delete_agent(v_name, force => true);

    PERFORM aidb.create_agent(v_name, 
        'You are a helpful, read-only database assistant. Use search_food to find the answers to questions about nutrition.',
        'claude_model_rag_over_documents',
        tools => ARRAY['search_food']);
END
$do$;

## Agent Conversation

Ask that same question, but to the agent. Instead of returning raw rows, the agent decides to call `search_food`, reads the passages that come back, and writes a natural-language answer based on them.

In [ ]:
from sqlalchemy import text
from IPython.display import Markdown, display
import pandas as pd

def ask(agent, question, conversation_id=None):
    """Ask the agent, render the answer, return (message, conversation_id).
    Pass a conversation_id from an earlier call to ask a follow-up."""
    with conn.begin() as c:
        df = pd.read_sql(
            text("""SELECT * FROM aidb.agent_converse(
                        CAST(:agent AS text), CAST(:q AS text),
                        conversation_id => CAST(:cid AS text))"""),
            c, params={"agent": agent, "q": question, "cid": conversation_id})
    row = df.iloc[0]
    if row["error"]:
        display(Markdown(f"**Agent error:** {row['error']}"))
    else:
        display(Markdown(row["message"]))
    # On an error the function returns a NULL id, so keep the one passed in
    new_id = row["conversation_id"]
    return row["message"], (conversation_id if pd.isna(new_id) else new_id)

AGENT = "demo_agent_nutrition"
_, conversation_id = ask(AGENT, "What should I eat to build muscle?")

# Part 2: Knowledge Base on PDF documents

Switching gears, let's upload PDF documents for the Knowledge Base, create a custom tool, and create an agent to ask questions.

Instead of creating an embedding per row in a table, we will parse PDF files, chunk them into smaller sections, and then create embeddings for each chunk. Splitting long documents into chunks improves the accuracy of finding the information inside them, because each embedding represents a focused passage rather than a whole publication. This is all done with an AI Pipeline.

The steps are: **storage location, volume, files, pipeline, search, tool, agent.**

## Allowed Local Paths
You can manage how the PGFS interacts with your local data with the `pgfs.allowed_local_fs_paths` configuration setting.

By default, PGFS is only able to access files in `/tmp/pgfs` of the local filesystem. This setting is a security measure to prevent the extension from accessing files it shouldn't.

To allow PGFS to access files in other locations, add those locations to the `pgfs.allowed_local_fs_paths` GUC. This parameter accepts a colon-separated list of file system paths that PGFS is allowed to access.

Example:
`ALTER SYSTEM SET pgfs.allowed_local_fs_paths = '/tmp/pgfs:/tmp/other';`

*Note: After changing this setting, you must restart the database.*

First, let's get the current value.

In [ ]:
#load extension into memory by getting the version.
%sql SELECT pgfs.pgfs_version();

result = %sql show pgfs.allowed_local_fs_paths
path = result[0][0]

storage_location = f"file://{path}/" 
print(storage_location)

## Create directory storage location

This example uses a local file location. The cell creates the folder on the database server where the PDFs will be stored.

In [ ]:
%%sql
SELECT pgfs.create_storage_location('local_pdfs', :storage_location);

## Create PGFS directory for a Volume

In [ ]:
!mkdir -p {path}/pdfs

## Create AIDB Volume

A **volume** presents the files in a storage location to AIDB as a source that pipelines can read. This one points at the `pdfs/` folder and treats its files as PDF documents. The Volume will be the source for an AI Pipeline.

The first statement deletes any earlier volume on this storage location so the cell can be re-run.

In [ ]:
%%sql
DO $do$
DECLARE
    v_schema text := 'public';
    v_name text := 'pdf_volume';
    v_fullname text := v_schema || '.' || v_name;
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.volumes WHERE schema = v_schema AND volume = v_name) THEN
        PERFORM aidb.delete_volume(v_fullname);
    END IF;

    PERFORM aidb.create_volume('pdf_volume', 'local_pdfs', 'pdfs/', 'Pdf');
END
$do$;


## Download demo files

These are 2025 IRS tax documents, downloaded directly into the folder registered for the storage location:

- **`i1040gi.pdf`**: the instructions for Form 1040.
- **`p17.pdf`**: Publication 17, *Your Federal Income Tax*.

They are long, real-world documents, which makes them a good test of chunking. The download runs on the database host, so it needs internet access. If the host is offline, copy the two files into the same folder manually instead.

In [ ]:
!mkdir -p {path}/pdfs
!curl https://www.irs.gov/pub/irs-pdf/i1040gi.pdf > {path}/pdfs/i1040gi.pdf
!curl https://www.irs.gov/pub/irs-pdf/p17.pdf > {path}/pdfs/p17.pdf

## Create the AI Pipeline

AI Pipelines can ingest, prepare, and vectorize data for semantic retrieval and RAG applications. They integrate pgvector search capabilities and can keep the knowledge base synchronized with your source data automatically.

This pipeline has three steps:

1. **`ParsePdf`** extracts the text from each PDF. `Structured` parsing preserves layout such as tables, and `allow_partial_parsing` lets the pipeline continue if part of a file cannot be read.
2. **`ChunkText`** splits the text into pieces of about 100 words (never more than 120), with 10 words of overlap between neighbouring chunks. This improves the accuracy of searches in large documents. The chunks are saved (`intermediate_destination`) so the matching text can be returned with each search hit.
3. **`KnowledgeBase`** creates a vector embedding for each chunk with the `bge-small-en-v1.5-f16` model.

`auto_processing => 'Disabled'` means the pipeline runs only when asked, and `aidb.run_pipeline` does that. Because the PDFs are large, **this step can take a few minutes**.

In [ ]:
%%sql

DO $do$
DECLARE
    v_name text := 'pdf_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.pipelines WHERE name = v_name) THEN
        PERFORM aidb.delete_pipeline(v_name);
    END IF;

    PERFORM aidb.create_pipeline(
        name           => v_name,
        source         => 'pdf_volume',
        step_1         => 'ParsePdf',
        step_1_options => aidb.pdf_parse_config(method => 'Structured', allow_partial_parsing => true),
        step_2         => 'ChunkText',
        step_2_options => aidb.chunk_text_config(desired_length => 100, max_length => 120, overlap_length => 10, strategy => 'words')
                          || jsonb_build_object('intermediate_destination', jsonb_build_object('enabled', true)),
        step_3         => 'KnowledgeBase',
        step_3_options => aidb.knowledge_base_config(model => 'bge-small-en-v1.5-f16', data_format => 'Text'),
        auto_processing => 'Disabled'
    );
    
    PERFORM aidb.run_pipeline(v_name);

END
$do$

## Example question

As before, we can execute similarity searches on the vector embeddings in the database, but the real power is when you create a custom tool and use an agent to ask questions about your data.

This query asks *"what are the new rules for taxes on tips?"* and returns the 8 closest chunks. For each hit you see the source file (`key`), the chunk's position (`part_ids`), a `distance` score, and the text of the chunk itself. `deduplicate => false` allows several chunks from the same file to appear.

In [ ]:
%%sql
SELECT key,
       part_ids,
       distance::numeric(5,3) AS dist,
       intermediate_steps -> 0 ->> 'value' AS chunk_text
FROM aidb.retrieve_text('public.pipeline_pdf_kb', --
                        'what are the new rules for taxes on tips?', --question
                        8, --distance
                        deduplicate => false)
ORDER BY 3 DESC;

## Create Custom Tool

This tool works just like the one in Part 1, but it searches the PDF knowledge base instead. It returns the source file, the distance, and the chunk text for each hit.

The `description` names the documents it covers (the 2025 Form 1040 instructions and Publication 17), so an agent that has several tools can tell when this is the right one to use. `${query}` and `${top_k}` are filled in by the agent.

In [ ]:
%%sql
DO $do$

DECLARE
    v_name text := 'search_tax_docs';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.sql_tool_registry WHERE name = v_name) THEN
        PERFORM aidb.delete_tool(v_name);
    END IF;
    
    PERFORM aidb.create_sql_tool(
        name          => v_name,
        description   => 'Semantic search over IRS 2025 Form 1040 instructions and Publication 17. Returns the most relevant passages with their source file. Use for any tax rules question.',
        sql_statement => $$
            SELECT key AS source_file,
                   distance::numeric(5,3) AS distance,
                   intermediate_steps -> 0 ->> 'value' AS passage
            FROM aidb.retrieve_text('public.pipeline_pdf_kb'::text, ${query}::text, ${top_k}::int, deduplicate => false)
        $$,
        params        => aidb.tool_params(
            aidb.tool_param('query', 'TEXT', 'The question or search phrase.'),
            aidb.tool_param('top_k', 'INT', 'Number of passages to return, e.g. 6.')
        ),
        read_only     => true
    );
END
$do$;


## Create new Agent for PDF files

This agent has only the `search_tax_docs` tool. Its instructions tell it to use the tool to find answers, so its answers come from the IRS documents and not from the model's general memory.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'demo_agent_tax_docs';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;
    
    PERFORM aidb.create_agent(v_name, 
        'You are a helpful, read-only database assistant. Use search_tax_docs to find the answers to questions.',
        'claude_model_rag_over_documents',
        tools => ARRAY['search_tax_docs']);
END
$do$;


### Ask the tax agent

Ask a question that is answered by the PDFs. The agent searches the documents, reads the most relevant passages, and summarizes the rules in plain language.

In [ ]:
AGENT = "demo_agent_tax_docs"
_, conversation_id = ask(AGENT, "What are the new rules for overtime?")

## Create an Agent that has both Custom Tools

Now combine the two sources. This agent has both `search_food` and `search_tax_docs`, and it can use either tool to find answers to questions. A realistic use case could be a subject-area agent with many tools for different sources of data. You can then ask the agent questions on the subject matter, and it figures out which tool to use to get the right answer.

How does it choose? It reads each tool's `description`, which is why clear descriptions matter.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'demo_agent_taxes_and_nutrition';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;

    PERFORM aidb.create_agent(v_name, 
        'You are a helpful, read-only database assistant. Use search_tax_docs and search_food to find the answers to questions.',
        'claude_model_rag_over_documents',
        tools => ARRAY['search_food', 'search_tax_docs']);
END
$do$;

## Ask the Agent with both Tools

The question is about IRS tax rules, so the agent should choose `search_tax_docs`. Tip: call `ask(..., debug=True)` to see more detail about which tool was used.

In [ ]:
AGENT = "demo_agent_taxes_and_nutrition"
_, conversation_id = ask(AGENT, "What are the new rules for overtime?")

## Ask the Agent with both Tools

This question is about nutrition, which is found in the database table. The same agent should now choose `search_food` instead. One agent, two very different sources of data, and no change to the question format.

In [ ]:
AGENT = "demo_agent_taxes_and_nutrition"
_, conversation_id = ask(AGENT, "What should I eat to build muscle?")

## Logging

With AIDB, the conversations are stored automatically in tables. This makes it easy to audit how the agents were used, identify problems, and make sure the agents were used in the intended way. Nothing needs to be configured to turn this on.

- `aidb.conversations` has one row per conversation.
- `aidb.conversation_log` has the individual steps of a conversation, in order.

Both queries below use the `conversation_id` of the most recent question asked in this notebook.

In [ ]:
%%sql
SELECT *
FROM aidb.conversations
where conversation_id = :conversation_id;

In [ ]:
%%sql
SELECT *
FROM aidb.conversation_log
WHERE conversation_id = :conversation_id
ORDER BY created_at;

## Recap

You built the same retrieval pattern twice, from two very different sources, and then combined them:

1. **Part 1**: one embedding per row of a table, with no chunking needed.
2. **Part 2**: PDFs parsed, chunked, and embedded by an AI Pipeline.
3. **Custom SQL tools** that give an agent controlled, read-only access to each knowledge base.
4. **A multi-tool agent** that picks the right source for each question.
5. **Automatic logging** of every conversation in ordinary Postgres tables.

**Where to go next**

- *Conversations Agent*: analyze the logs you just looked at.
- *Governance*: tie an agent to a database role so it only sees data it is allowed to see.
- *MCP - Claude Desktop Integration*: expose the same tax search tool to Claude Desktop.